# Titanic Survival Prediction

This notebook builds a machine learning pipeline for Titanic passenger survival prediction using tabular preprocessing and classification.




#### Build a Titanic survival prediction pipeline using the cleaned dataset.

- Split the cleaned data into training and test sets

- Define a pipeline to execute column_transformer and then svc

- Perform GritSearchCV() with the pipeline and 5 folds cross-validation for each combination of parameter

- The parameter combination is

  * C = 0.001, 0.01, 0.1, 1, 10, 100
  * gamma = 0.001, 0.01, 0.1, 1, 10, 100

- Print out the following
  * Best parameter combination
  * Best score for the training data
  * Best score for the test data

- Use and modify the hint programs (mentioned in the lecture)


v Hint programs: Use and modify these and complete the program


In [ ]:
import pandas as pd

# URL of the Titanic dataset on GitHub
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

# Read the Titanic dataset from the URL
df = pd.read_csv(url)

# Display the initial information about the dataset
print("Before cleaning:")
print(df.info())
print("\nMissing values (by column):")
print(df.isnull().sum())

# Drop 'Cabin' column becase it has too many missing values
df.drop(columns=['Cabin'], inplace=True)

# Drop rows with missing values in 'Age' and 'Embarked' columns
df.dropna(subset=['Age', 'Embarked'], inplace=True)

# Save the cleansing dataset to a CSV file
df.to_csv("titanic_cleaned.csv", index=False)
print("\nCleansed dataset saved as 'titanic_cleaned.csv'")



Before cleaning:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
None

Missing values (by column):
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cab

In [ ]:
# Read the cleansed dataset from the CSV file
df_cleansed = pd.read_csv("titanic_cleaned.csv")

# Show the cleaned dataset information
print("\nAfter removing 'Cabin' column:")
print(df_cleansed.info())


After removing 'Cabin' column:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 712 entries, 0 to 711
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  712 non-null    int64  
 1   Survived     712 non-null    int64  
 2   Pclass       712 non-null    int64  
 3   Name         712 non-null    object 
 4   Sex          712 non-null    object 
 5   Age          712 non-null    float64
 6   SibSp        712 non-null    int64  
 7   Parch        712 non-null    int64  
 8   Ticket       712 non-null    object 
 9   Fare         712 non-null    float64
 10  Embarked     712 non-null    object 
dtypes: float64(2), int64(5), object(4)
memory usage: 61.3+ KB
None


In [ ]:
# Pick up ['sex', 'embarked', 'class', 'age', 'fare'] as X
X = df_cleansed[['Sex', 'Embarked', 'Pclass', 'Age', 'Fare']]
# Pick up 'survived' as y
y = df_cleansed['Survived']

# print the shape of the features
print("\nShape of features (X):", X.shape)
print("Head of features (X):\n", X.head())


Shape of features (X): (712, 5)
Head of features (X):
       Sex Embarked  Pclass   Age     Fare
0    male        S       3  22.0   7.2500
1  female        C       1  38.0  71.2833
2  female        S       3  26.0   7.9250
3  female        S       1  35.0  53.1000
4    male        S       3  35.0   8.0500


In [ ]:
#define the column transformer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
# Define the column transformer to handle categorical and numerical features
column_transformer = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['Age', 'Fare']),
        ('cat', OneHotEncoder(), ['Sex', 'Embarked', 'Pclass']) # Categorical features
    ],
    remainder='passthrough'  # Keep other columns as they are
)
# Apply the column transformer to the features
X_transformed = column_transformer.fit_transform(X)
# Print the shape of the transformed features
print("\nShape of transformed features (X_transformed):", X_transformed.shape)
# Print the feature names after transformation
feature_names = column_transformer.get_feature_names_out()
print("Feature names after transformation:\n", feature_names)
print("Head of transformed features (X_transformed):\n", X_transformed[:5])


Shape of transformed features (X_transformed): (712, 10)
Feature names after transformation:
 ['num__Age' 'num__Fare' 'cat__Sex_female' 'cat__Sex_male'
 'cat__Embarked_C' 'cat__Embarked_Q' 'cat__Embarked_S' 'cat__Pclass_1'
 'cat__Pclass_2' 'cat__Pclass_3']
Head of transformed features (X_transformed):
 [[-0.52766856 -0.51637992  0.          1.          0.          0.
   1.          0.          0.          1.        ]
 [ 0.57709388  0.69404605  1.          0.          1.          0.
   0.          1.          0.          0.        ]
 [-0.25147795 -0.50362035  1.          0.          0.          0.
   1.          0.          0.          1.        ]
 [ 0.36995092  0.35032585  1.          0.          0.          0.
   1.          1.          0.          0.        ]
 [ 0.36995092 -0.50125747  0.          1.          0.          0.
   1.          0.          0.          1.        ]]


In [ ]:
# Fill the program here

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


pipeline = Pipeline(steps=[
    ('preprocess', column_transformer),
    ('svc', SVC())
])

param_grid = {
    'svc__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'svc__gamma': [0.001, 0.01, 0.1, 1, 10, 100]
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, n_jobs=-1)
grid_search.fit(X_train, y_train)


y_pred = grid_search.best_estimator_.predict(X_test)


print("\nBest parameter combination:")
print(grid_search.best_params_)

print("\nBest score for the training data (cross-validation score):")
print(grid_search.best_score_)

print("\nBest score for the test data (test accuracy):")
print(accuracy_score(y_test, y_pred))


Best parameter combination:
{'svc__C': 10, 'svc__gamma': 0.1}

Best score for the training data (cross-validation score):
0.7978885266263003

Best score for the test data (test accuracy):
0.8041958041958042
